# Libros durante cuarentena
## Introducción

Muchas personas se quedaron en casa durante una pandemia, eso llevó que algunos dedicaran su tiempo a leer libros, por lo cual una *startup* inició el desarrollo de una aplicación para los amantes de los libros, con el propósito de generar una propuesta de valor para un nuevo producto

## Desarrollo

Para eso se importaron las librerías necesarias, y se conectó a la base de datos para comenzar las consultas

In [4]:
# importar librerías
import pandas as pd
from sqlalchemy import create_engine


db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',  # Sin espacio
    'port': 5432,
    'db': 'data-analyst-final-project-db'
}         

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

In [5]:
engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

In [6]:
query_test = """

SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public';

"""
tablas = pd.io.sql.read_sql(query_test, con=engine)
print("Tablas disponibles:")
print(tablas)

Tablas disponibles:
           table_name
0             ratings
1  advertisment_costs
2             authors
3              orders
4             reviews
5              visits
6               books
7               users
8          publishers


In [7]:
import socket
try:
    socket.gethostbyname('yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com')
    print("Host resuelto correctamente")
except socket.gaierror:
    print("No se puede resolver el host")

Host resuelto correctamente


### Número de libros
Se hizo un conteo por la cantidad de libros publicados después de 1 de enero de 200: Se encontraron 819 libros, se reveló que la base de datos tiene una buena representación de literatura moderna, la plataforma se enfocan en lectores interesados

In [8]:
query = """

SELECT COUNT(*) AS libros_despues_2000
FROM books
WHERE publication_date > '2000-01-01';

"""

# Ejecutar la consulta
resultado = pd.io.sql.read_sql(query, con=engine)
print("Número de libros publicados después del 1 de enero de 2000:")
print(resultado)

Número de libros publicados después del 1 de enero de 2000:
   libros_despues_2000
0                  819


### Total de reseñas

Se hizo un conteo del total de reseñas; se encontraron 2793 reseñas, buena parte de los libros se han reseñado en promedio **2.8**, y por lo general, la calificación es entre **3.5 y 4.5**, lo que puede indicar que los usuarios califican principalmente libros que les gustaron, o que la selección de libros en la plataforma es de buena calidad.. Los usuarios están dispuestos a compartir sus opiniones, aunque hay potencial para aumentar dicha interacción. Aunque la mayoría de los libros han recibido al menos una reseña, mostrando una buena cobertura, la distribución varía mucho 

In [12]:
query = """
SELECT COUNT(*) AS reseñas_usuarios
FROM reviews
"""

# Ejecutar la consulta
resultado = pd.io.sql.read_sql(query, con=engine)
print("Número de reseñas de usuarios:")
print(resultado)

Número de reseñas de usuarios:
   reseñas_usuarios
0              2793


In [23]:
query = """

SELECT 
    book_id,
    COUNT(*) AS numero_reseña
FROM reviews
GROUP BY book_id;

"""

# Ejecutar la consulta
resultado = pd.io.sql.read_sql(query, con=engine)
print("Número de reseñas de usuarios para cada libro:")
print(resultado)

Número de reseñas de usuarios para cada libro:
     book_id  numero_reseña
0        652              2
1        273              2
2         51              5
3        951              2
4        839              4
..       ...            ...
989       64              4
990       55              2
991      148              3
992      790              2
993      828              2

[994 rows x 2 columns]


In [22]:
query = """


SELECT 
    book_id,
    AVG(rating) AS calificacion_promedio
FROM ratings
GROUP BY book_id;


"""

# Ejecutar la consulta
resultado = pd.io.sql.read_sql(query, con=engine)
print("Número de calificación promedio para cada libro:")
print(resultado)

Número de calificación promedio para cada libro:
     book_id  calificacion_promedio
0        652               4.500000
1        273               4.500000
2         51               4.250000
3        951               4.000000
4        839               4.285714
..       ...                    ...
995       64               4.230769
996       55               5.000000
997      148               3.428571
998      790               3.500000
999      828               3.000000

[1000 rows x 2 columns]


### Editoriales

Se han contado las editoriales que han publicado libros con más de 50 páginas, en las cuales van entre 10 y 40 libros publicados. Hay una concentración significativa en pocas editoriales grandes. Las top 5 editoriales publican entre 19-42 libros, mientras que muchas otras solo tienen 1-2 libros, reflejando la estructura real del mercado editorial.

In [24]:
query = """



SELECT 
    publisher_id,
    COUNT(*) AS numero_libros 
FROM books
WHERE num_pages > 50
GROUP BY publisher_id
ORDER BY numero_libros DESC;


"""


# Ejecutar la consulta
resultado = pd.io.sql.read_sql(query, con=engine)
print("Editoriales que han publicado el mayor número de libros:")
print(resultado)

Editoriales que han publicado el mayor número de libros:
     publisher_id  numero_libros
0             212             42
1             309             31
2             116             25
3             217             24
4              33             19
..            ...            ...
329            34              1
330           225              1
331           138              1
332           245              1
333           205              1

[334 rows x 2 columns]


### Autores

Se han contado la cantidad des autores con la calificación más alta, se usaron como referencia al menos, 50 calificaciones para entrar al umbral, entre los que destacan: 

1. Diana Gabaldon
2. J.K. Rowling/Mary GrandPré
3. Agatha Christie
4. Markus Zusak/Cao Xuân Việt Khương
5. J.R.R. Tolkien

Los autores con mejores calificaciones son principalmente escritores establecidos y populares.

In [56]:
query = """

SELECT
    author,
    AVG(rating) AS calificacion_promedio_autor 
FROM ratings r 
JOIN books b ON r.book_id = b.book_id 
JOIN authors a ON b.author_id = a.author_id
GROUP BY author
HAVING COUNT(*) >= 50
ORDER BY calificacion_promedio_autor  DESC;

"""


# Ejecutar la consulta
resultado = pd.io.sql.read_sql(query, con=engine)
print("Autores con la más alta calificación promedio del libro:")
print(resultado)

Autores con la más alta calificación promedio del libro:
                                               author  \
0                                      Diana Gabaldon   
1                          J.K. Rowling/Mary GrandPré   
2                                     Agatha Christie   
3                   Markus Zusak/Cao Xuân Việt Khương   
4                                      J.R.R. Tolkien   
5                            Roald Dahl/Quentin Blake   
6                                   Louisa May Alcott   
7                                        Rick Riordan   
8                                       Arthur Golden   
9                                        Stephen King   
10                                       John Grisham   
11                                    William Golding   
12                                    Nicholas Sparks   
13                                       Jodi Picoult   
14                                    Sophie Kinsella   
15                             

### Reseñas de texto y usuarios activos

Se analizaron las reseñas de texto, se dividió en 2, una para identificar a los usuarios que calificaron más de 50 libros, y después se calculó el número promedio, en la que escriben en promedio 24.33 reseñas de texto.
Solo el 0.6% de usuarios son extremadamente activos, esto muestra que los usuarios más comprometidos no solo califican muchos libros, sino que también escriben reseñas detalladas. Esto los convierte en creadores de contenido clave para atraer y guiar a otros lectores.

In [80]:

query = """

SELECT
    username,
    COUNT(*) as num_calificaciones
FROM ratings
GROUP BY username
HAVING COUNT(*) > 50

"""


# Ejecutar la consulta
resultado = pd.io.sql.read_sql(query, con=engine)
print("Número de usuarios que calificaron más de 50 libros:")
print(resultado)


Número de usuarios que calificaron más de 50 libros:
         username  num_calificaciones
0     sfitzgerald                  55
1  jennifermiller                  53
2          xdavis                  51
3          paul88                  56
4      martinadam                  56
5       richard89                  55


In [79]:
query = """SELECT AVG(num_reseñas)

FROM (
    SELECT username, COUNT(*) as num_reseñas
    FROM reviews
    WHERE username IN (
        SELECT
            username
        FROM ratings
        GROUP BY username
    HAVING COUNT(*) > 50
    )
    GROUP BY username
) AS subconsulta

"""

# Ejecutar la consulta
resultado = pd.io.sql.read_sql(query, con=engine)
print("Número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros:")
print(resultado)

Número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros:
         avg
0  24.333333


## Conclusiones

Tras las consultas, se reveló que la plataforma tiene un núcleo pequeño pero muy valioso de usuarios súper comprometidos que generan contenido de alta calidad, lo que puede ser la base para estrategias de retención y crecimiento. Las recomendaciones para la empresa son en crear un programa de "Embajadores de Lectura" para estos usuarios súper activos, ofreciendo:

* Acceso anticipado a nuevos libros
* Insignias especiales en sus perfiles
* Posibilidad de moderar comunidades temáticas

Tambien pueden desarrollar secciones especializadas como "Nuevos Lanzamientos" y "Tendencias Actuales" para capitalizar este inventario moderno. Los autores mejor calificados son principalmente escritores establecidos, esto puede servir para crear "Colecciones de Autor" y sistemas de recomendación basados en estos autores populares.